<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">ספייק 2: <span dir="ltr">torch</span> ב-<span dir="ltr">Colab</span> עם GPU</p>
<p dir="rtl" style="direction:rtl; text-align:right;">מחברת בדיקה למחבר. סביבת ריצה: <b dir="ltr">Runtime → Change runtime type → R</b> ו-<b dir="ltr">T4 GPU</b>. הריצו תא אחרי תא (לא <span dir="ltr">Run all</span>, כדי לראות איפה משהו נכשל) ורשמו את הזמנים שמודפסים. זמן כולל משוער: 15 עד 25 דקות.</p>


In [ ]:
# Timing helper: every step records its duration in a table printed at the end
timings <- data.frame(step = character(), seconds = numeric(), note = character())
tick <- function(step, expr, note = "") {
  t0 <- Sys.time()
  on.exit({
    s <- round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1)
    timings[nrow(timings) + 1, ] <<- list(step, s, note)
    cat(sprintf("[%s] %.1f s\n", step, s))
  })
  force(expr)
}
cat(R.version.string, "\n")
system("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
system("nvcc --version | tail -1")   # system CUDA toolkit (may be absent)

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">1. התקנת <span dir="ltr">torch</span></p>
<p dir="rtl" style="direction:rtl; text-align:right;">ההתקנה הרגילה מורידה את ספריית <span dir="ltr">libtorch/Lantern</span> (כ-2 ג׳יגה-בייט עם <span dir="ltr">CUDA</span>). אם התא נכשל, ראו את תא הגיבוי מתחתיו.</p>


In [ ]:
# Plan A: plain install from CRAN; TORCH_INSTALL=1 makes it download libtorch non-interactively
options(timeout = 900)
Sys.setenv(TORCH_INSTALL = "1")
tick("install.packages(torch)", install.packages("torch"))
tick("library(torch) + download libtorch", library(torch))
cat("torch version:", as.character(packageVersion("torch")), "\n")
cat("CUDA available:", cuda_is_available(), "\n")

In [ ]:
# Plan B (run ONLY if cuda_is_available() is FALSE or the install failed):
# pre-built binaries with bundled CUDA from the mlverse CDN (kind = "cu128" per the torch docs)
# options(timeout = 900)
# kind <- "cu128"
# version <- "0.16.0"   # <- set to the current CRAN version of torch
# options(repos = c(mlverse = sprintf("https://torch-cdn.mlverse.org/packages/%s/%s/", kind, version),
#                   CRAN = "https://cloud.r-project.org"))
# tick("install torch cu128", install.packages("torch"))
# Plan C: install the native libraries manually
# tick("install_torch", torch::install_torch(timeout = 900))

In [ ]:
tick("install luz + torchvision + helpers",
     install.packages(c("luz", "torchvision", "readr", "dplyr", "stringr", "jpeg", "tibble")))
library(luz); library(torchvision); library(dplyr); library(readr); library(stringr)
device <- if (cuda_is_available()) "cuda" else "cpu"
cat("device:", device, "\n")
x <- torch_randn(2000, 2000, device = device)
tick("2000x2000 matmul x50 on device", for (i in 1:50) y <- torch_mm(x, x), note = device)

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">2. כיוונון <span dir="ltr">ResNet-18</span> על 72 תמונות של עזרי ניידות</p>
<p dir="rtl" style="direction:rtl; text-align:right;">הרשימה (<span dir="ltr">manifest</span>) נמצאת בקובץ <code dir="ltr">mobility_aids_manifest.csv</code> (בספר: <code dir="ltr">book_src/data/</code>). העלו אותו לסביבת העבודה (סמל התיקייה משמאל, העלאה), או הגדירו כתובת הורדה ב-<code dir="ltr">MANIFEST_URL</code>. התמונות מורדות מ-<span dir="ltr">Wikimedia Commons</span>.</p>


In [ ]:
# Get the manifest: upload it by hand in the Files pane, or set MANIFEST_URL (raw GitHub link)
MANIFEST_URL <- "https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/data/mobility_aids_manifest.csv"
if (!file.exists("mobility_aids_manifest.csv") && nzchar(MANIFEST_URL)) {
  download.file(MANIFEST_URL, "mobility_aids_manifest.csv")
}
stopifnot(file.exists("mobility_aids_manifest.csv"))
aids <- read_csv("mobility_aids_manifest.csv", show_col_types = FALSE) |>
  mutate(local = file.path("aids", sprintf("%s_%02d.jpg", class, row_number())), .by = class)
dir.create("aids", showWarnings = FALSE)
ua <- c("User-Agent" = "DSAI4PT-course-spike/0.1 (teaching use)")
tick("download 72 images", {
  for (i in seq_len(nrow(aids))) {
    if (!file.exists(aids$local[i])) {
      url <- paste0("https://commons.wikimedia.org/wiki/Special:FilePath/",
                    URLencode(str_replace_all(aids$file[i], " ", "_"), reserved = TRUE), "?width=320")
      try(download.file(url, aids$local[i], mode = "wb", headers = ua, quiet = TRUE))
      Sys.sleep(0.3)   # be polite to Wikimedia
    }
  }
})
ok <- file.exists(aids$local) & file.size(aids$local) > 1000
cat("downloaded:", sum(ok), "of", nrow(aids), "\n")
aids <- aids[ok, ]

In [ ]:
# Load every image as a 3x224x224 normalized tensor (ImageNet mean/std)
load_img <- function(path) {
  img <- jpeg::readJPEG(path)                       # height x width x channels in 0..1
  if (length(dim(img)) == 2) img <- array(rep(img, 3), c(dim(img), 3))   # gray -> RGB
  t <- torch_tensor(img[, , 1:3], dtype = torch_float())$permute(c(3, 1, 2))
  t <- nnf_interpolate(t$unsqueeze(1), size = c(224, 224), mode = "bilinear")$squeeze(1)
  m <- torch_tensor(c(0.485, 0.456, 0.406))$view(c(3, 1, 1))
  s <- torch_tensor(c(0.229, 0.224, 0.225))$view(c(3, 1, 1))
  (t - m) / s
}
set.seed(10)
aids <- aids |> mutate(split = if_else(row_number() %in% sample(n(), 8), "test", "train"), .by = class)
levels_ <- sort(unique(aids$class))
x_all <- torch_stack(lapply(aids$local, load_img))
y_all <- torch_tensor(match(aids$class, levels_), dtype = torch_long())
tr <- which(aids$split == "train"); te <- which(aids$split == "test")
dim(x_all)

In [ ]:
# Fine-tune ResNet-18 (last stage + new output layer), 3 epochs, plain torch loop
torch_manual_seed(1)
net <- NULL
tick("load pretrained ResNet-18", net <- model_resnet18(pretrained = TRUE))
for (p in net$parameters) p$requires_grad_(FALSE)
for (p in net$layer4$parameters) p$requires_grad_(TRUE)
net$fc <- nn_linear(512, length(levels_))
net$to(device = device)
trainable <- Filter(function(p) p$requires_grad, net$parameters)
opt <- optim_adam(trainable, lr = 1e-4)
xd <- x_all$to(device = device); yd <- y_all$to(device = device)
acc_hist <- c()
tick("fine-tune 3 epochs", {
  for (epoch in 1:3) {
    net$train()
    for (b in split(sample(tr), rep(1:3, length.out = length(tr)))) {
      opt$zero_grad()
      loss <- nnf_cross_entropy(net(xd[b, , , ]), yd[b])
      loss$backward(); opt$step()
    }
    net$eval()
    pred <- with_no_grad(net(xd[te, , , ]))$argmax(dim = 2)
    acc <- as.numeric((pred == yd[te])$to(dtype = torch_float())$mean())
    acc_hist <- c(acc_hist, acc)
    cat(sprintf("epoch %d  loss %.3f  test acc %.2f\n", epoch, loss$item(), acc))
  }
}, note = device)

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">3. משימת טקסט קטנה ל-<span dir="ltr">A5</span>: סיווג משפטים קליניים קצרים</p>
<p dir="rtl" style="direction:rtl; text-align:right;">הצעה: 4 קטגוריות (כאב, ניידות, תרופות, מצב רוח), כ-40 משפטים באנגלית (כאן 24 לבדיקה; סינתטיים, בלי מידע על מטופלים אמיתיים). כל משפט הופך להטבעה (<span dir="ltr">embedding</span>) של מודל מאומן מראש, ועל גביה מאמנים שכבת פלט קטנה ב-<span dir="ltr">torch</span>. זו אותה לוגיקה של העברת למידה כמו בתמונות. אם חישוב ההטבעות החי נכשל, יש גיבוי: קובץ הטבעות מחושב מראש.</p>


In [ ]:
sentences <- tibble::tribble(
  ~text, ~label,
  "The patient reports sharp pain in the right shoulder when lifting the arm", "pain",
  "Pain increases at night and wakes her up", "pain",
  "He rates the knee pain as seven out of ten", "pain",
  "Burning sensation along the left forearm after exercise", "pain",
  "Constant aching in the lower back during walking", "pain",
  "Pain is relieved by rest and a cold pack", "pain",
  "She walks ten meters with a four wheeled walker and supervision", "mobility",
  "Needs moderate assistance to transfer from bed to wheelchair", "mobility",
  "Able to climb one flight of stairs holding the rail", "mobility",
  "Gait speed improved to point eight meters per second", "mobility",
  "He stands up from the chair without using his hands", "mobility",
  "Uses a cane outdoors and walks independently at home", "mobility",
  "Take one tablet of baclofen twice a day after meals", "medication",
  "The dose of the blood thinner was increased last week", "medication",
  "She forgot to take the morning pills on two days", "medication",
  "Botulinum toxin injection planned for the calf muscle", "medication",
  "Side effects include drowsiness and dry mouth", "medication",
  "Stopped the antidepressant without consulting the doctor", "medication",
  "He feels discouraged and avoids meeting friends", "mood",
  "She is anxious about falling again", "mood",
  "Motivated and cheerful during the therapy session", "mood",
  "Reports low mood and poor sleep since the stroke", "mood",
  "He was irritable and refused to continue the exercises", "mood",
  "She says she feels hopeful about returning to work", "mood")
sentences$id <- seq_len(nrow(sentences))
table(sentences$label)

In [ ]:
# Option 1 (live): sentence embeddings with python sentence-transformers via reticulate
emb <- NULL
tick("embeddings via reticulate", {
  try({
    system("pip -q install sentence-transformers")
    if (!requireNamespace("reticulate", quietly = TRUE)) install.packages("reticulate")
    st <- reticulate::import("sentence_transformers")
    m <- st$SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    emb <- m$encode(sentences$text, normalize_embeddings = TRUE)   # n x 384 matrix
  })
}, note = "needs python bridge")
if (is.null(emb)) cat("Option 1 failed -> use Option 2 (pre-computed CSV)\n") else dim(emb)

In [ ]:
# Option 2 (fallback): pre-computed embeddings, one row per sentence (id, label, e1..e384)
# TODO: generate once in python and host next to the notebook (clinical_sentences_emb.csv)
if (is.null(emb) && file.exists("clinical_sentences_emb.csv")) {
  pre <- read_csv("clinical_sentences_emb.csv", show_col_types = FALSE)
  emb <- as.matrix(pre[, grep("^e[0-9]+$", names(pre))])
  sentences <- pre[, c("id", "label")]
}
# Option 3 (tests the torch/GPU path only, no pretrained knowledge): hashed bag of words
if (is.null(emb)) {
  cat("Using hashed bag-of-words ONLY to test the torch GPU path\n")
  words <- strsplit(tolower(gsub("[^a-z ]", "", sentences$text)), " +")
  emb <- t(sapply(words, function(w) {
    v <- numeric(256)
    for (x in w) { u <- utf8ToInt(x); i <- sum(u * seq_along(u)) %% 256 + 1; v[i] <- v[i] + 1 }
    v / sqrt(sum(v^2))
  }))
}
y <- match(sentences$label, sort(unique(sentences$label)))
set.seed(3)
te_t <- unlist(lapply(split(seq_along(y), y), function(i) sample(i, 2)))   # 2 test sentences per class
tr_t <- setdiff(seq_along(y), te_t)
X <- torch_tensor(emb, dtype = torch_float(), device = device)
Y <- torch_tensor(y, dtype = torch_long(), device = device)
torch_manual_seed(1)
head <- nn_linear(ncol(emb), length(unique(y)))$to(device = device)
opt <- optim_adam(head$parameters, lr = 0.01)
tick("train text head 200 steps", {
  for (s in 1:200) {
    opt$zero_grad(); loss <- nnf_cross_entropy(head(X[tr_t, ]), Y[tr_t]); loss$backward(); opt$step()
  }
})
pred <- head(X[te_t, ])$argmax(dim = 2)
text_acc <- as.numeric((pred == Y[te_t])$to(dtype = torch_float())$mean())
cat("text test accuracy:", round(text_acc, 2), "on", length(te_t), "held-out sentences\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">4. סיכום זמנים</p>
<p dir="rtl" style="direction:rtl; text-align:right;">העתיקו את הפלט של התא הבא ושלחו אותו. אם משהו נכשל, העתיקו גם את הודעת השגיאה.</p>


In [ ]:
print(timings)
cat("\nTOTAL seconds:", sum(timings$seconds), "\n")
cat("CUDA:", cuda_is_available(), "| torch", as.character(packageVersion("torch")),
    "| image acc by epoch:", paste(round(acc_hist, 2), collapse = ", "),
    "| text acc:", round(text_acc, 2), "\n")